# roadstyle — output types

roadstyle takes the **same styled road data** and gives it to you in several **output forms**, from a ready-to-open map object to a stack-agnostic JSON spec you can embed in any website. This notebook shows each one, what it returns, and when to use it.

| Output | Function | Returns | Use when |
|---|---|---|---|
| Folium map | `render_edges(...)` | `folium.Map` | notebook / a standalone HTML file |
| Lonboard map | `render_edges(..., backend='lonboard')` | `lonboard.Map` | very large data (WebGL) |
| Canonical JSON | `to_spec(...)` | `dict` | feed your own web frontend |
| Styled GeoJSON | `to_geojson(...)` | `dict` | you only need the FeatureCollection |
| Full HTML page | `to_html(..., full=True)` / `save(...)` | `str` / file | a complete page |
| HTML fragment | `to_html(..., full=False)` | `str` | inject into an existing page |
| Iframe embed | `to_iframe(...)` | `str` | drop into a page, **zero JS** |

> Run top-to-bottom (Kernel → Restart & Run All). Files are written to `outputs_guide_out/`.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import geopandas as gpd
import roadstyle as rs

print('roadstyle', rs.__version__)
edges = gpd.read_file('data/sundbyberg_edges.gpkg')   # 4039 real OSM road edges
OUT = Path('outputs_guide_out'); OUT.mkdir(exist_ok=True)
print('edges:', len(edges), '| columns:', list(edges.columns))

## 1. A map object — `folium.Map` (the default)

`render_edges` returns a live `folium.Map`. In a notebook you can display it by ending a cell with the object; on disk you `.save()` it to a standalone interactive HTML file.

In [ ]:
m = rs.render_edges(edges, theme='dark')
print('returns:', type(m).__module__ + '.' + type(m).__name__)
m.save(str(OUT / '1_folium.html'))
print('saved', (OUT / '1_folium.html').name)
# m   # <- uncomment in a live notebook to view inline

## 2. A WebGL map object — `lonboard.Map`

Pass `backend='lonboard'` for a GPU-rendered map (deck.gl) — smooth for very large edge sets. Same styling engine, different rendering tech. Save with `.to_html()`.

In [ ]:
try:
    gpu = rs.render_edges(edges, backend='lonboard', theme='dark')
    print('returns:', type(gpu).__module__ + '.' + type(gpu).__name__, '| layers:', len(gpu.layers))
    gpu.to_html(str(OUT / '2_lonboard.html'))
    print('saved', (OUT / '2_lonboard.html').name)
except ImportError:
    print('lonboard not installed — pip install lonboard (or the [lonboard] extra)')

## 3. The canonical JSON spec — `to_spec` (the web handoff)

`to_spec` returns a plain `dict` (JSON) containing the normalized data **plus the baked-in resolved style**: each feature carries `__rs_fill`, `__rs_w`, `__rs_casing`, … so any frontend just reads and draws — it doesn't need roadstyle's logic. This is the stack-agnostic output.

In [ ]:
spec = rs.to_spec(edges, color_by='maxspeed', cmap='viridis', theme='dark', width_by=(1, 6))
print('top-level keys:', sorted(spec.keys()))
print('version:', spec['roadstyle'], '| crs:', spec['crs'], '| theme:', spec['theme'])
f0 = spec['geojson']['features'][0]['properties']
print('per-feature style props:', {k: f0[k] for k in f0 if k.startswith('__rs_')})
print('legend:', {k: spec['legend'][k] for k in ('kind', 'title', 'vmin', 'vmax')})

Write it to a `.json` file (serve this to a browser, or reload it with `load_spec`):

In [ ]:
rs.save_spec(edges, str(OUT / '3_roads.json'), color_by='maxspeed', cmap='viridis', width_by=(1, 6))
back = rs.load_spec(str(OUT / '3_roads.json'))
print('round-trip OK:', back['roadstyle'] == spec['roadstyle'],
      '| features:', len(back['geojson']['features']))

## 4. Styled GeoJSON only — `to_geojson`

If you just want the `FeatureCollection` (features still carry the `__rs_*` style props):

In [ ]:
gj = rs.to_geojson(edges, color_by='maxspeed', cmap='viridis')
print('type:', gj['type'], '| features:', len(gj['features']))
print('has __rs_fill:', '__rs_fill' in gj['features'][0]['properties'])

## 5. HTML — full page vs fragment — `to_html`

`full=True` → a complete, self-contained HTML page (Leaflet auto-loaded). `full=False` → just a `<div>+<script>` you paste into an existing page's body.

In [ ]:
page = rs.to_html(edges, color_by='maxspeed', cmap='viridis', full=True)
frag = rs.to_html(edges, color_by='maxspeed', cmap='viridis', full=False)
print('full page :', 'starts <!DOCTYPE' if page.lstrip().startswith('<!DOCTYPE') else '?',
      '| KB:', len(page) // 1000)
print('fragment  :', 'starts <div' if frag.lstrip().startswith('<div') else '?',
      '| no <!DOCTYPE:', '<!DOCTYPE' not in frag)
(OUT / '5_fragment.html').write_text(frag)
print('saved', (OUT / '5_fragment.html').name)

## 6. Iframe embed — `to_iframe` (zero front-end code)

The simplest way to put a map on a page: a self-contained `<iframe srcdoc=…>` string. Paste it anywhere — no JavaScript, no separate file.

In [ ]:
ifr = rs.to_iframe(edges, color_by='maxspeed', cmap='viridis', height='500px')
print('starts <iframe:', ifr.startswith('<iframe'), '| has srcdoc:', 'srcdoc=' in ifr)
print(ifr[:90] + ' …')

## 7. Standalone file — `save`

`save` writes a complete interactive HTML map in one call (the spec-based renderer — lighter than folium's `.save()` for the same data).

In [ ]:
rs.save(edges, str(OUT / '7_standalone.html'), color_by='maxspeed', cmap='viridis', width_by=(1, 6))
import os
spec_kb = os.path.getsize(OUT / '7_standalone.html') // 1000
folium_kb = os.path.getsize(OUT / '1_folium.html') // 1000
print(f'spec HTML: {spec_kb} KB   vs   folium HTML: {folium_kb} KB')

## Recap

All of `outputs_guide_out/` came from the **same edges**, just different output forms:

- **Just want a map?** `render_edges(...)` → folium (or `backend='lonboard'`).
- **Embed in a page, no code?** `to_iframe(...)` or `save(...)` + an `<iframe src=...>`.
- **Your own web frontend?** `to_spec(...)` / `save_spec(...)` → JSON, then render with the Leaflet/MapLibre snippets in `docs/embedding.md`.

See **docs/embedding.md** (frontend snippets) and **docs/frontend.md** (baked HTML vs JSON API vs a JS port) for the next step.